In [64]:
import sqlite3

def update_experiment_view(db_path='../cem_results.db'):
    """
    Drops the existing experiment_summary view and recreates it 
    with the updated metrics schema including the testset column.
    """
    # Define the SQL script
    sql_script = """
    DROP VIEW IF EXISTS experiment_summary;

    CREATE VIEW experiment_summary AS
    SELECT 
        r.dataset,
        r.entity, 
        r.model_type,
        r.train_size, 
        r.seed, 
        r.lm,
        m.pollution, 
        m.iteration, 
        m.testset,
        m.f1_score,
        m.precision,
        m.recall,
        m.is_final,
        r.run_id,
        r.timestamp
    FROM metrics m
    JOIN runs r ON m.run_id = r.run_id;
    """

    try:
        # Establish connection
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        
        # Execute the script (handles multiple statements)
        cursor.executescript(sql_script)
        
        conn.commit()
        print("View 'experiment_summary' has been updated successfully.")
        
    except sqlite3.Error as e:
        print(f"An error occurred: {e}")
        
    finally:
        if conn:
            conn.close()

# Run the update


In [65]:
import sqlite3
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Connection
conn = sqlite3.connect('../cem_results.db')
update_experiment_view()

target_entity = "movie"
target_size = 1.0
target_seed = 42
target_pollution = "high"


try:

    summary_query = f"""
        SELECT * FROM experiment_summary 
        WHERE entity = '{target_entity}' AND
        train_size = '{target_size}' AND
        seed = {target_seed} AND 
        pollution = '{target_pollution}'
        ORDER BY "timestamp" DESC
    """
    
    # Execute and load into DataFrame
    df_results = pd.read_sql_query(summary_query, conn)
    display(df_results)

finally:
    conn.close()


results_filtered = df_results.drop(columns=["dataset", "model_type", "lm", "testset", "is_final", "run_id", "timestamp"])
print(results_filtered.to_csv())

View 'experiment_summary' has been updated successfully.


,dataset,entity,model_type,train_size,seed,lm,pollution,iteration,testset,f1_score,precision,recall,is_final,run_id,timestamp
0,imdb,movie,baseline,1.0,42,roberta,high,0,test,0.828,0.945,0.738,1,9763648_3_105948_movie,2026-04-16 10:59:48
1,imdb,movie,baseline,1.0,42,roberta,high,1,test,0.828,0.945,0.738,1,9763648_3_105948_movie,2026-04-16 10:59:48
2,imdb,movie,1,1.0,42,roberta,high,0,scoring,0.875,0.972,0.797,0,9763556_3_101113_movie,2026-04-16 10:11:13
3,imdb,movie,1,1.0,42,roberta,high,0,conv,0.868,0.969,0.786,0,9763556_3_101113_movie,2026-04-16 10:11:13
4,imdb,movie,1,1.0,42,roberta,high,1,scoring,0.974,0.972,0.977,0,9763556_3_101113_movie,2026-04-16 10:11:13
5,imdb,movie,1,1.0,42,roberta,high,1,conv,0.906,0.945,0.871,0,9763556_3_101113_movie,2026-04-16 10:11:13
6,imdb,movie,1,1.0,42,roberta,high,2,scoring,0.978,0.977,0.979,0,9763556_3_101113_movie,2026-04-16 10:11:13
7,imdb,movie,1,1.0,42,roberta,high,2,conv,0.907,0.945,0.872,0,9763556_3_101113_movie,2026-04-16 10:11:13
8,imdb,movie,1,1.0,42,roberta,high,3,scoring,0.976,0.966,0.986,0,9763556_3_101113_movie,2026-04-16 10:11:13
9,imdb,movie,1,1.0,42,roberta,high,3,conv,0.910,0.945,0.878,1,9763556_3_101113_movie,2026-04-16 10:11:13


,entity,train_size,seed,pollution,iteration,f1_score,precision,recall
0,movie,1.0,42,high,0,0.828,0.945,0.738
1,movie,1.0,42,high,1,0.828,0.945,0.738
2,movie,1.0,42,high,0,0.875,0.972,0.797
3,movie,1.0,42,high,0,0.868,0.969,0.786
4,movie,1.0,42,high,1,0.974,0.972,0.977
5,movie,1.0,42,high,1,0.906,0.945,0.871
6,movie,1.0,42,high,2,0.978,0.977,0.979
7,movie,1.0,42,high,2,0.907,0.945,0.872
8,movie,1.0,42,high,3,0.976,0.966,0.986
9,movie,1.0,42,high,3,0.91,0.945,0.878
10,movie,1.0,42,high,0,0.975,0.959,0.992
11,movie,1.0,42,high,0,0.984,0.974,0.995
12,movie,1.0,42,high,1,0.975,0.959,0.992
13,movie,1.0,42,high,1,0.984,0.974,0.995
14,movie,1.0,42,high,2,0.975,0.959,0.992
15,movie,1.0,42,high,2,0.984,0.974,0.995
16,movie,1.0,42,high,0,0.983,0.981,0.986
17,movie,1.0,42,high,0,0.989,0.991,0.988
18,movie,1.0,42,high,1,0.982,0.98,0.983
19,movie,1.0,42,high,1,0.989,0.991,0.987
20,movie,1.0,42,high,0,0.979,0.967,0.992
21,movie,1.0,42,high,0,0.981,0.971,0.992
22,movie,1.0,42,high,0,0.823,0.727,0.948

In [66]:


import sqlite3
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Connection
conn = sqlite3.connect('../cem_results.db')
update_experiment_view()
i = 8 # Set your desired number of recent runs here

try:
    summary_query = f"""
        SELECT * FROM experiment_summary WHERE model_type = 'baseline' ORDER BY "timestamp"  desc LIMIT 20
    """
    
    # Execute and load into DataFrame
    df_baseline = pd.read_sql_query(summary_query, conn)
    
    print(f"Successfully retrieved data for {len(df_results)} runs.")
    baseline_filtered = df_baseline.drop(columns=["dataset", "lm", "testset", "is_final", "run_id", "timestamp"])
    display(baseline_filtered)

finally:
    conn.close()

# print(df_baseline.to_csv())

View 'experiment_summary' has been updated successfully.
Successfully retrieved data for 55 runs.


,entity,model_type,train_size,seed,pollution,iteration,f1_score,precision,recall
0,name,baseline,1.0,30,high,0,0.917,0.953,0.883
1,name,baseline,1.0,30,high,0,0.020,0.021,0.019
2,name,baseline,1.0,30,high,0,0.026,0.250,0.014
3,name,baseline,1.0,30,high,1,0.460,0.313,0.864
4,name,baseline,1.0,30,high,1,0.898,0.935,0.864
5,name,baseline,1.0,30,high,2,0.454,0.311,0.840
6,name,baseline,1.0,30,high,2,0.883,0.933,0.837
7,name,baseline,1.0,30,high,3,0.460,0.311,0.883
8,name,baseline,1.0,30,high,3,0.909,0.937,0.883
9,name,baseline,1.0,20,high,0,0.917,0.953,0.883


In [67]:
target_entity = "movie"
target_size = 1.0
target_seed = 42
target_pollution = "high"

# Filter the dataframe
mask = (
    # (results_filtered["entity"] == target_entity) &
    (results_filtered["train_size"] == target_size) &
    (results_filtered["seed"] == target_seed) &
    (results_filtered["pollution"] == target_pollution)
)

display(results_filtered[mask])

# Filter the dataframe
mask = (
    # (results_filtered["entity"] == target_entity) &
    (baseline_filtered["train_size"] == target_size) &
    (baseline_filtered["seed"] == target_seed) &
    (baseline_filtered["pollution"] == target_pollution)
)

display(baseline_filtered[mask])



import pandas as pd

# Define the columns to match on
match_cols = ["train_size", "seed", "pollution", "entity"]

# Perform an inner join to align the rows
df_comparison = pd.merge(
    df_results, 
    df_baseline, 
    on=match_cols, 
    suffixes=('_res', '_base')
)

display(df_comparison)
# Now you can calculate differences, for example:
df_comparison['score_diff'] = df_comparison['f1_score_res'] - df_comparison['f1_score_base']
display(df_comparison)

,entity,train_size,seed,pollution,iteration,f1_score,precision,recall
0,movie,1.0,42,high,0,0.828,0.945,0.738
1,movie,1.0,42,high,1,0.828,0.945,0.738
2,movie,1.0,42,high,0,0.875,0.972,0.797
3,movie,1.0,42,high,0,0.868,0.969,0.786
4,movie,1.0,42,high,1,0.974,0.972,0.977
5,movie,1.0,42,high,1,0.906,0.945,0.871
6,movie,1.0,42,high,2,0.978,0.977,0.979
7,movie,1.0,42,high,2,0.907,0.945,0.872
8,movie,1.0,42,high,3,0.976,0.966,0.986
9,movie,1.0,42,high,3,0.910,0.945,0.878


,entity,model_type,train_size,seed,pollution,iteration,f1_score,precision,recall


,dataset_res,entity,model_type_res,train_size,seed,lm_res,pollution,iteration_res,testset_res,f1_score_res,...,model_type_base,lm_base,iteration_base,testset_base,f1_score_base,precision_base,recall_base,is_final_base,run_id_base,timestamp_base


,dataset_res,entity,model_type_res,train_size,seed,lm_res,pollution,iteration_res,testset_res,f1_score_res,...,lm_base,iteration_base,testset_base,f1_score_base,precision_base,recall_base,is_final_base,run_id_base,timestamp_base,score_diff
